In [1]:
!pip install -U transformers accelerate sentence-transformers pinecone-client langchain langchain-community langchain-text-splitters

   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   --- ------------------------------------ 0.8/10.1 MB 5.6 MB/s eta 0:00:02
   ------- -------------------------------- 1.8/10.1 MB 4.4 MB/s eta 0:00:02
   ---------- ----------------------------- 2.6/10.1 MB 3.9 MB/s eta 0:00:02
   ------------- -------------------------- 3.4/10.1 MB 4.0 MB/s eta 0:00:02
   ---------------- ----------------------- 4.2/10.1 MB 3.9 MB/s eta 0:00:02
   ------------------- -------------------- 5.0/10.1 MB 3.9 MB/s eta 0:00:02
   ---------------------- ----------------- 5.8/10.1 MB 3.9 MB/s eta 0:00:02
   ------------------------- -------------- 6.6/10.1 MB 3.9 MB/s eta 0:00:01
   ----------------------------- ---------- 7.3/10.1 MB 3.9 MB/s eta 0:00:01
   -------------------------------- ------- 8.1/10.1 MB 3.8 MB/s eta 0:00:01
   ----------------------------------- ---- 8.9/10.1 MB 3.8 MB/s eta 0:00:01
   -------------------------------------- - 9.7/10.1 MB 3.8 MB/s eta 0:00:01
   ---

In [1]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("pineconedata.txt")
documents = loader.load()

documents

[Document(metadata={'source': 'pineconedata.txt'}, page_content='Artificial Intelligence (AI) is a field of computer science focused on building systems that can perform tasks requiring human intelligence. These tasks include learning, reasoning, problem-solving, and understanding language.\n\nMachine Learning (ML) is a subset of AI that allows systems to learn from data instead of being explicitly programmed. Common types include supervised learning, unsupervised learning, and reinforcement learning.\n\nDeep Learning is a subset of machine learning that uses neural networks with many layers. It is widely used in image recognition, speech processing, and natural language processing.\n\nNatural Language Processing (NLP) enables machines to understand and generate human language. Applications include chatbots, translation systems, and sentiment analysis.\n\nTransformers are a type of neural network architecture that rely on self-attention mechanisms. They are the foundation of modern lar

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

len(docs)

13

In [3]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\Piyush\AppData\Local\Temp\ipykernel_3468\2985419238.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
sample_vec = embedding.embed_query("test")
dim = len(sample_vec)

print("Embedding Dimension:", dim)

Embedding Dimension: 384


In [23]:
!pip uninstall pinecone-client -y

In [24]:
!pip uninstall pinecone -y

Found existing installation: pinecone 7.3.0
Uninstalling pinecone-7.3.0:
  Successfully uninstalled pinecone-7.3.0


In [25]:
!pip uninstall langchain-pinecone -y

Found existing installation: langchain-pinecone 0.2.13
Uninstalling langchain-pinecone-0.2.13:
  Successfully uninstalled langchain-pinecone-0.2.13


In [26]:
!pip install pinecone
!pip install langchain-pinecone

  Using cached pinecone-8.1.0-py3-none-any.whl.metadata (14 kB)
  Using cached pinecone_plugin_assistant-3.0.2-py3-none-any.whl.metadata (30 kB)
Using cached pinecone-8.1.0-py3-none-any.whl (742 kB)
Using cached pinecone_plugin_assistant-3.0.2-py3-none-any.whl (280 kB)

  Attempting uninstall: pinecone-plugin-assistant

    Found existing installation: pinecone-plugin-assistant 1.8.0

    Uninstalling pinecone-plugin-assistant-1.8.0:

      Successfully uninstalled pinecone-plugin-assistant-1.8.0

   ---------------------------------------- 0/2 [pinecone-plugin-assistant]
   ---------------------------------------- 0/2 [pinecone-plugin-assistant]
   ---------------------------------------- 0/2 [pinecone-plugin-assistant]
   ---------------------------------------- 0/2 [pinecone-plugin-assistant]
   ---------------------------------------- 0/2 [pinecone-plugin-assistant]
   ---------------------------------------- 0/2 [pinecone-plugin-assistant]
   --------------------------------------

In [5]:
import os

os.environ["PINECONE_API_KEY"] = "xyz"

In [35]:
from pinecone import Pinecone

pc = Pinecone()  # no api_key needed now

In [7]:
from langchain_pinecone import PineconeVectorStore

index_name = "test"

In [8]:
vectordb = PineconeVectorStore.from_documents(
    documents=docs,
    embedding=embedding,
    index_name=index_name,
    namespace="default"
)

In [9]:
query = "What is RAG?"

results = vectordb.similarity_search(query, k=2)

for doc in results:
    print(doc.page_content)
    print("-" * 50)

RAG (Retrieval-Augmented Generation) is a technique where relevant documents are retrieved and passed to a language model to generate better answers.

LangChain is a framework for building applications with LLMs. It provides tools for chaining models, retrieval, and memory.
--------------------------------------------------
Pinecone is a managed vector database used for storing and searching embeddings efficiently.

Chroma is a lightweight vector database often used for local development and experimentation.

Weaviate is an open-source vector database that supports hybrid search combining keyword and vector search.
--------------------------------------------------


In [10]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "xyz"

In [31]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="gpt2",
    max_new_tokens=100,   
    do_sample=True,
    temperature=0.5
)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

C:\Users\Piyush\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Piyush\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [32]:
from langchain_community.llms import HuggingFacePipeline

llm = HuggingFacePipeline(pipeline=pipe)

In [33]:
from langchain_classic.chains import RetrievalQA

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectordb.as_retriever()
)

In [34]:
qa.invoke("What is RAG?")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'query': 'What is RAG?',
 'result': "Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\nRAG (Retrieval-Augmented Generation) is a technique where relevant documents are retrieved and passed to a language model to generate better answers.\n\nLangChain is a framework for building applications with LLMs. It provides tools for chaining models, retrieval, and memory.\n\nPinecone is a managed vector database used for storing and searching embeddings efficiently.\n\nChroma is a lightweight vector database often used for local development and experimentation.\n\nWeaviate is an open-source vector database that supports hybrid search combining keyword and vector search.\n\nRegularization techniques like L1 and L2 help prevent overfitting by penalizing large weights.\n\nAPIs (Application Programming Interfaces) allow different software systems to communicate with each other.\n\nBERT